# Power BI report fix: the numbers

This notebook is the source of truth for every number in the README, the diagrams and `powerbi/06-checks.md`, and it writes them there itself.

1. Build the data both reports read (`data/sales_flat.csv`).
2. Compute, with plain SQL, every number each card, chart and table must show. The slow report and the fixed report must both match these.
3. Read the timings and model sizes measured in Power BI Desktop (`measurements/`) and work out how much faster and smaller the fixed report is.
4. Write every number into `powerbi/06-checks.md`, `README.md` and the diagrams in `docs/`.

Run it top to bottom from a fresh clone: `pip install -r requirements.txt`, then `jupyter nbconvert --to notebook --execute --inplace analysis/analysis.ipynb`.

In [1]:
%run ../prepare_data.py

Wrote data/sales_flat.csv (2,098,633 rows, 801 MB)


In [2]:
from datetime import date
from pathlib import Path
import re

import duckdb
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)
ROOT = Path("..")
FLAT = ROOT / "data" / "sales_flat.csv"

con = duckdb.connect()
# Only the columns the report uses, to keep memory low.
con.sql(f'''
CREATE TABLE sales AS
SELECT "Order Key", "Order Date", "Customer Key", "Country", "Product Name", "Category", "Subcategory",
       "Quantity", "Net Price", "Unit Cost"
FROM '{FLAT}'
''')
q = lambda sql: con.sql(sql).df()

flat_columns = len(con.sql(f"SELECT * FROM '{FLAT}' LIMIT 0").columns)
overview = q('''
SELECT count(*)                        AS sales_rows,
       count(DISTINCT "Order Key")     AS orders,
       count(DISTINCT "Customer Key")  AS customers,
       count(DISTINCT "Product Name")  AS products,
       min("Order Date")               AS first_order,
       max("Order Date")               AS last_order
FROM sales
''')
overview

,sales_rows,orders,customers,products,first_order,last_order
0,2098633,875901,88063,2517,2015-01-01,2024-04-20


## What the report must show

The same measures as the report, written in SQL. Values are rounded the way the report shows them.

| Measure | Rule |
|---|---|
| Sales Amount | sum of Quantity × Net Price |
| Margin % | (Sales Amount − sum of Quantity × Unit Cost) ÷ Sales Amount |
| Orders, Customers | distinct order keys, distinct customer keys |
| Avg Order Value | Sales Amount ÷ Orders |
| Sales PY | Sales Amount for the same months one year earlier, with every other filter kept |
| Sales YoY % | (Sales Amount − Sales PY) ÷ Sales PY |
| Product Rank | rank of the product's Sales Amount among the products in the current filters, ties share a rank |

`kpis(where_now, where_py)` returns the six cards for any slicer selection.

In [3]:
def kpis(where_now, where_py):
    return q(f'''
    WITH now AS (
        SELECT sum("Quantity" * "Net Price")                     AS sales,
               sum("Quantity" * ("Net Price" - "Unit Cost"))     AS margin,
               count(DISTINCT "Order Key")                       AS orders,
               count(DISTINCT "Customer Key")                    AS customers
        FROM sales WHERE {where_now}
    ), py AS (
        SELECT sum("Quantity" * "Net Price") AS sales_py FROM sales WHERE {where_py}
    )
    SELECT round(sales)                                      AS "Sales Amount",
           round(100 * (sales - sales_py) / sales_py, 1)     AS "Sales YoY %",
           round(100 * margin / sales, 1)                    AS "Margin %",
           orders                                            AS "Orders",
           round(sales / orders, 2)                          AS "Avg Order Value",
           customers                                         AS "Customers",
           round(sales_py)                                   AS "Sales PY"
    FROM now, py
    ''')

# Page 1, default selection: Year 2023, every country, every category.
cards_default = kpis('year("Order Date") = 2023', 'year("Order Date") = 2022')
cards_default

,Sales Amount,Sales YoY %,Margin %,Orders,Avg Order Value,Customers,Sales PY
0,"318,425,878.00",-28.40,56.00,159695,"1,993.96",62337,"444,516,719.00"


In [4]:
# Page 1, a filtered selection: Year 2022, Country Germany, Category Computers.
cards_filtered = kpis('''year("Order Date") = 2022 AND "Country" = 'Germany' AND "Category" = 'Computers' ''',
                      '''year("Order Date") = 2021 AND "Country" = 'Germany' AND "Category" = 'Computers' ''')
cards_filtered

,Sales Amount,Sales YoY %,Margin %,Orders,Avg Order Value,Customers,Sales PY
0,"19,645,197.00",93.50,56.00,9854,"1,993.63",5197,"10,153,937.00"


In [5]:
# Page 1 chart: Sales Amount and Sales PY by month, Year 2023.
by_month = q('''
SELECT strftime(min("Order Date"), '%b') AS "Month",
       round(sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2023)) AS "Sales Amount",
       round(sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2022)) AS "Sales PY"
FROM sales
WHERE year("Order Date") IN (2022, 2023)
GROUP BY month("Order Date") ORDER BY month("Order Date")
''')
by_month

,Month,Sales Amount,Sales PY
0,Jan,"34,197,837.00","36,052,336.00"
1,Feb,"40,800,178.00","49,140,854.00"
2,Mar,"22,604,720.00","29,700,003.00"
3,Apr,"11,939,474.00","19,221,969.00"
4,May,"28,979,019.00","44,587,592.00"
5,Jun,"27,454,645.00","46,105,321.00"
6,Jul,"23,332,124.00","34,978,428.00"
7,Aug,"24,849,603.00","35,682,325.00"
8,Sep,"25,110,286.00","36,801,749.00"
9,Oct,"25,116,723.00","36,552,588.00"


In [6]:
# Page 1 bars: Sales Amount by Category and by Country, Year 2023.
by_category = q('''
SELECT "Category", round(sum("Quantity" * "Net Price")) AS "Sales Amount"
FROM sales WHERE year("Order Date") = 2023 GROUP BY 1 ORDER BY 2 DESC
''')
by_country = q('''
SELECT "Country", round(sum("Quantity" * "Net Price")) AS "Sales Amount"
FROM sales WHERE year("Order Date") = 2023 GROUP BY 1 ORDER BY 2 DESC
''')
display(by_category, by_country)

,Category,Sales Amount
0,Computers,"113,112,204.00"
1,Cell phones,"59,754,615.00"
2,Home Appliances,"54,115,079.00"
3,TV and Video,"41,323,827.00"
4,"Music, Movies and Audio Books","21,768,661.00"
5,Cameras and camcorders,"18,765,677.00"
6,Audio,"6,847,569.00"
7,Games and Toys,"2,738,246.00"


,Country,Sales Amount
0,United States,"156,132,838.00"
1,Germany,"41,279,015.00"
2,Canada,"37,241,921.00"
3,United Kingdom,"26,252,830.00"
4,Australia,"22,522,330.00"
5,Netherlands,"17,031,456.00"
6,France,"11,734,079.00"
7,Italy,"6,231,408.00"


In [7]:
# Page 2 table: every product sold in 2023 by Sales Amount; the top 10 go in the checks.
products = q('''
WITH p AS (
    SELECT "Product Name",
           sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2023)                       AS sales,
           sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2022)                       AS sales_py,
           sum("Quantity" * ("Net Price" - "Unit Cost")) FILTER (WHERE year("Order Date") = 2023)       AS margin,
           count(DISTINCT "Customer Key") FILTER (WHERE year("Order Date") = 2023)                      AS customers
    FROM sales WHERE year("Order Date") IN (2022, 2023)
    GROUP BY 1
)
SELECT rank() OVER (ORDER BY sales DESC)             AS "Product Rank",
       "Product Name",
       round(sales)                                  AS "Sales Amount",
       round(sales_py)                               AS "Sales PY",
       round(100 * (sales - sales_py) / sales_py, 1) AS "Sales YoY %",
       round(100 * margin / sales, 1)                AS "Margin %",
       customers                                     AS "Customers"
FROM p WHERE sales IS NOT NULL
ORDER BY sales DESC
''')
print(len(products), "products in the table")
products.head(10)

2517 products in the table


,Product Rank,Product Name,Sales Amount,Sales PY,Sales YoY %,Margin %,Customers
0,1,"Adventure Works 52"" LCD HDTV X590 White","2,317,817.00","2,712,426.00",-14.50,64.80,262
1,2,"Adventure Works 52"" LCD HDTV X590 Brown","2,264,428.00","2,867,452.00",-21.00,64.70,270
2,3,"Adventure Works 52"" LCD HDTV X590 Silver","2,038,838.00","3,218,644.00",-36.70,65.00,228
3,4,"Adventure Works 52"" LCD HDTV X590 Black","1,804,751.00","3,159,420.00",-42.90,64.30,207
4,5,Adventure Works Desktop PC2.33 XD233 White,"1,803,958.00","3,444,403.00",-47.60,64.80,621
5,6,Adventure Works Desktop PC2.33 XD233 Brown,"1,775,896.00","3,110,374.00",-42.90,64.60,623
6,7,WWI Desktop PC2.33 X2330 Silver,"1,733,712.00","3,152,473.00",-45.00,65.00,607
7,8,Adventure Works Desktop PC2.33 XD233 Silver,"1,712,446.00","3,158,969.00",-45.80,64.90,634
8,9,WWI Desktop PC2.33 X2330 Black,"1,678,976.00","3,077,120.00",-45.40,64.90,627
9,10,WWI Desktop PC2.33 X2330 Brown,"1,657,784.00","3,108,136.00",-46.70,64.80,615


In [8]:
# Page 2 matrix: Category rows with the grand total, Year 2023.
category_matrix = q('''
SELECT coalesce("Category", 'Total') AS "Category",
       round(sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2023)) AS "Sales Amount",
       round(100 * (sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2023)
                  - sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2022))
                  / sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2022), 1) AS "Sales YoY %",
       round(100 * sum("Quantity" * ("Net Price" - "Unit Cost")) FILTER (WHERE year("Order Date") = 2023)
                 / sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2023), 1) AS "Margin %"
FROM sales WHERE year("Order Date") IN (2022, 2023)
GROUP BY ROLLUP ("Category")
ORDER BY "Category" IS NULL, 2 DESC
''')
category_matrix

,Category,Sales Amount,Sales YoY %,Margin %
0,Computers,"113,112,204.00",-36.10,56.20
1,Cell phones,"59,754,615.00",-27.70,54.00
2,Home Appliances,"54,115,079.00",-16.40,55.30
3,TV and Video,"41,323,827.00",-28.10,56.90
4,"Music, Movies and Audio Books","21,768,661.00",-22.30,58.60
5,Cameras and camcorders,"18,765,677.00",-21.10,58.20
6,Audio,"6,847,569.00",-10.40,55.10
7,Games and Toys,"2,738,246.00",-13.20,52.00
8,Total,"318,425,878.00",-28.40,56.00


In [9]:
# The slow report's Sales PY takes the year from the rows in each cell (MAX of Sales[Year]), so it matches the
# fixed report only where every cell shown has current-year sales. Count the cells in the checked states that don't: all must be 0.
empty_cells = q('''
WITH s AS (SELECT * FROM sales WHERE year("Order Date") IN (2021, 2022, 2023))
SELECT
  (SELECT 12 - count(DISTINCT month("Order Date")) FROM s WHERE year("Order Date") = 2023) AS months_without_2023_sales,
  (SELECT 12 - count(DISTINCT month("Order Date")) FROM s
    WHERE year("Order Date") = 2022 AND "Country" = 'Germany' AND "Category" = 'Computers') AS months_without_sales_filtered_state,
  (SELECT count(DISTINCT "Subcategory") FROM sales)
    - (SELECT count(DISTINCT "Subcategory") FROM s WHERE year("Order Date") = 2023)   AS subcategories_without_2023_sales,
  (SELECT count(*) FROM (SELECT DISTINCT "Product Name" FROM s WHERE year("Order Date") = 2022
                         EXCEPT SELECT DISTINCT "Product Name" FROM s WHERE year("Order Date") = 2023)) AS products_sold_2022_not_2023
''')
assert (empty_cells.iloc[0] == 0).all(), "a checked cell has no current-year sales: the two reports would differ there"
empty_cells

,months_without_2023_sales,months_without_sales_filtered_state,subcategories_without_2023_sales,products_sold_2022_not_2023
0,0,0,0,0


## Before and after: timings and model size

Measured in Power BI Desktop by following `measurements/README.md`: one Performance Analyzer export per page per report, and one VertiPaq Analyzer export (`.vpax`, from DAX Studio) per report.

The cells that read them are added once the exports are in `measurements/`. They fill `timings`: the before and after numbers, plus a headline, card title and card result that lead with the speed-up. The last cell writes them like every other number.

In [10]:
timings = {}  # empty until the Power BI exports are in measurements/

## Write the numbers into the repo

`powerbi/06-checks.md` is written in full. In `README.md` and `docs/*.svg`, each number sits between `<!--n:name-->` and `<!--/n-->` markers, and the value from `numbers` replaces whatever is between them. A marker with no value yet keeps its placeholder.

In [11]:
money = lambda v: f"${v:,.0f}"
money2 = lambda v: f"${v:,.2f}"
pct = lambda v: f"{v:.1f}%"
whole = lambda v: f"{int(v):,}"
FORMATS = {"Sales Amount": money, "Sales PY": money, "Sales YoY %": pct, "Margin %": pct,
           "Orders": whole, "Customers": whole, "Avg Order Value": money2}

def md_table(df):
    lines = ["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(FORMATS.get(c, str)(row[c]) for c in df.columns) + " |")
    return "\n".join(lines)

def cards_table(cards):
    row = cards.iloc[0]
    names = ["Sales Amount", "Sales YoY %", "Margin %", "Orders", "Avg Order Value", "Customers"]
    return md_table(pd.DataFrame({"Card": names, "Shows": [FORMATS[n](row[n]) for n in names]}))

o = overview.iloc[0]
d = cards_default.iloc[0]
date_rows = (date(o.last_order.year, 12, 31) - date(o.first_order.year, 1, 1)).days + 1
model_rows = pd.DataFrame({
    "Table": ["Sales (both reports)", "Customer (fixed report)", "Product (fixed report)", "Date (fixed report)"],
    "Rows (Table view, bottom left)": [whole(o.sales_rows), whole(o.customers), whole(o.products), whole(date_rows)],
})

checks = f'''# 6. Checks: the numbers each page must show

Written by [analysis/analysis.ipynb](../analysis/analysis.ipynb) from SQL on `data/sales_flat.csv`; do not edit by hand. **Both reports, the slow one and the fixed one, must show exactly these numbers.** If one differs, the build has a mistake; fix it before taking any timing.

The slow report's `Sales PY` takes the year from the rows in each cell, so it agrees with the fixed report only where every cell shown has sales in the selected year. The notebook confirms the check states below have no empty cells (every month, subcategory and product shown has current-year sales).

Small rounding note: a value ending in exactly .5 can round one unit differently in Power BI than in Python. Anything else is a real difference.

## Model

{md_table(model_rows)}

## Page 1: Overview, default state (Year 2023, all countries, all categories)

{cards_table(cards_default)}

Sales YoY % compares with Sales PY = {money(d["Sales PY"])} (2022).

**Sales and previous year by month** (hover a column to see both values):

{md_table(by_month)}

**Sales by category** (bar order, top to bottom):

{md_table(by_category)}

**Sales by country** (bar order, top to bottom):

{md_table(by_country)}

## Page 1: Overview, filtered (Year 2022, Country Germany, Category Computers)

Set these three slicers, check the cards, then set the slicers back to the default state.

{cards_table(cards_filtered)}

## Page 2: Products, default state (Year 2023)

**Products by sales:** the table has {whole(len(products))} rows (every product sold in 2023). The top 10:

{md_table(products.head(10))}

**Sales by category** matrix, collapsed to Category:

{md_table(category_matrix)}
'''
(ROOT / "powerbi" / "06-checks.md").write_text(checks, encoding="utf-8")

numbers = {
    "sales_rows": whole(o.sales_rows),
    "rows_million": f"{o.sales_rows / 1e6:.1f} million",
    "orders": whole(o.orders),
    "first_month": o.first_order.strftime("%B %Y"),
    "last_month": o.last_order.strftime("%B %Y"),
    "flat_columns": str(flat_columns),
    "archive_mb": f"{(ROOT / 'data' / 'csv-1m.7z').stat().st_size / 1e6:,.0f} MB",
    "flat_mb": f"{FLAT.stat().st_size / 1e6:,.0f} MB",
    "orders_2023": whole(d["Orders"]),
    "sales_2023": money(d["Sales Amount"]),
    "yoy_2023": pct(abs(d["Sales YoY %"])),
    "margin_2023": pct(d["Margin %"]),
    "headline": f"{whole(o.sales_rows)} sales rows: a {flat_columns}-column flat table rebuilt as a star schema, every number unchanged.",
    "headline_short": f"{whole(o.sales_rows)} sales rows, the same numbers before and after.",
    "card_title": "Rebuilding a slow sales report without changing a single number",
    "card_result": f"{whole(o.sales_rows)} sales rows: one {flat_columns}-column table rebuilt as a star schema of 4 tables, every number matched against SQL before and after.",
}
numbers.update(timings)

def fill(path):
    text = path.read_text(encoding="utf-8")
    text = re.sub(r"<!--n:(\w+)-->(.*?)<!--/n-->",
                  lambda m: f"<!--n:{m[1]}-->{numbers.get(m[1], m[2])}<!--/n-->", text, flags=re.S)
    path.write_text(text, encoding="utf-8", newline="\n")

for path in ["README.md", "docs/header.svg", "docs/before-after.svg"]:
    fill(ROOT / path)

# The portfolio site's card for this project, when the portfolio repo sits next to this one (Omar's laptop).
CARD = ROOT.resolve().parent / "portfolio" / "content" / "projects" / "powerbi-report-fix.en.md"
if CARD.exists():
    text = CARD.read_text(encoding="utf-8")
    text = re.sub(r'^title: ".*"$', f'title: "{numbers["card_title"]}"', text, flags=re.M)
    text = re.sub(r'^result: ".*"$', f'result: "{numbers["card_result"]}"', text, flags=re.M)
    CARD.write_text(text, encoding="utf-8", newline="\n")
    fill(CARD)
numbers

{'sales_rows': '2,098,633',
 'rows_million': '2.1 million',
 'orders': '875,901',
 'first_month': 'January 2015',
 'last_month': 'April 2024',
 'flat_columns': '35',
 'archive_mb': '42 MB',
 'flat_mb': '801 MB',
 'orders_2023': '159,695',
 'sales_2023': '$318,425,878',
 'yoy_2023': '28.4%',
 'margin_2023': '56.0%',
 'headline': '2,098,633 sales rows: a 35-column flat table rebuilt as a star schema, every number unchanged.',
 'headline_short': '2,098,633 sales rows, the same numbers before and after.',
 'card_title': 'Rebuilding a slow sales report without changing a single number',
 'card_result': '2,098,633 sales rows: one 35-column table rebuilt as a star schema of 4 tables, every number matched against SQL before and after.'}